1. Business Questions for the Columns

is_valid_purchase: "Did this order result in an actual sale, rather than being canceled or unavailable?"
is_approved: "Did the customer's payment successfully go through?"
is_shipped / is_delivered: "Did the product actually leave the warehouse and reach the customer?"
is_on_time: "Are our logistics partners keeping their delivery timeline promises to the customer?"
promised_days: "How long are we telling customers they have to wait for their items?"
item_value / freight_value: "How much revenue did the merchandise generate, and how much did it cost to ship it?"
is_good_order: "Was the customer experience flawless (on time and highly rated)?"
is_first_purchase: "Is this a new customer we just acquired, or a returning customer?"


2. Decisions vs Facts Answers

1. GMV = SUM(price) (excluding freight)

Alternative rejected: Using payment_value (which was used in Project 1).
Why: payment_value includes shipping costs (freight), and potentially discounts or interest. We want GMV to reflect the strict value of the goods sold, not the total transaction cost including logistics.
2. A "good order" = delivered on time AND reviewed 4–5★

Alternative rejected: Assuming no review means "everything was fine" and counting it as a good order.
Why: We are choosing a very strict, conservative definition of quality. If an order was never reviewed, we don't know for sure if the customer was happy. We are sacrificing quantity (it will look like we have fewer good orders) to guarantee that every "good order" was explicitly confirmed by the customer.
3. A "valid purchase" excludes canceled and unavailable

Alternative rejected: Including all orders regardless of final status.
Why: We are excluding them because we are calculating final realized sales. A canceled order doesn't generate real revenue. However, if we were analyzing the sales funnel (how many people tried to checkout) or retention (did a user interact with us again), we would count canceled orders, because a cancellation is still a strong signal of purchase intent.


In [2]:
import sys
sys.path.append("..")

from src.metrics import connect_raw, build_order_facts, MARTS, START, END

con = connect_raw()
facts = build_order_facts(con)
print(f"{len(facts):,} orders, {facts['customer_unique_id'].nunique():,} customers")
print(facts.dtypes)

99,092 orders, 95,774 customers
order_id                          str
customer_unique_id                str
customer_state                    str
order_status                      str
purchase_date          datetime64[us]
purchase_month         datetime64[us]
is_valid_purchase                bool
is_approved                      bool
is_shipped                       bool
is_delivered                     bool
is_on_time                       bool
promised_days                   int64
item_value                    float64
freight_value                 float64
n_items                         Int64
n_sellers                       Int64
main_category                     str
review_score                    Int64
is_good_order                    bool
first_purchase_date    datetime64[us]
is_first_purchase                bool
dtype: object


In [3]:
assert facts["order_id"].is_unique, "join fan-out: an order appears twice"

assert (facts["is_good_order"] <= facts["is_on_time"]).all()
assert (facts["is_on_time"] <= facts["is_delivered"]).all()




In [4]:
direct = con.execute(f"""
    SELECT SUM(oi.price)
    FROM order_items oi JOIN orders o ON oi.order_id = o.order_id
    WHERE CAST(o.order_purchase_timestamp AS DATE) BETWEEN DATE '{START}' AND DATE '{END}'
""").fetchone()[0]
print(f"facts GMV {facts['item_value'].sum():,.0f} vs direct {direct:,.0f}")
assert abs(facts["item_value"].sum() - direct) < 1

facts GMV 13,541,713 vs direct 13,541,713


In [5]:
con.execute("DESCRIBE category_translation").df()

,column_name,column_type,null,key,default,extra
0,product_category_name,VARCHAR,YES,None,None,None
1,product_category_name_english,VARCHAR,YES,None,None,None


In [6]:
print((facts["main_category"] == "unknown").mean(), "share unknown")
print(facts["main_category"].value_counts().head(10))

0.02168691720825092 share unknown
main_category
bed_bath_table           9325
health_beauty            8757
sports_leisure           7664
computers_accessories    6652
furniture_decor          6279
housewares               5812
watches_gifts            5602
telephony                4169
auto                     3869
toys                     3845
Name: count, dtype: int64


In [7]:
delivered_orders = facts[facts["is_delivered"] == True]
on_time_rate = delivered_orders["is_on_time"].mean()

print(f"Project 3 On-time rate: {on_time_rate:.2%} (vs Project 1: 93.2%)")

Project 3 On-time rate: 93.21% (vs Project 1: 93.2%)


In [8]:
valid_purchases = facts[facts["is_valid_purchase"] == True]
days_per_customer = valid_purchases.groupby("customer_unique_id")["purchase_date"].nunique()
is_repeat_customer = (days_per_customer > 1)
repeat_rate = is_repeat_customer.mean()
print(f"Project 3 Repeat rate: {repeat_rate:.2%} (vs Project 1: 2.16%)")

Project 3 Repeat rate: 2.17% (vs Project 1: 2.16%)


Reconcilation with project1 ad-hocs and general testing for metrics.


In [9]:
valid = facts[facts["is_valid_purchase"]]
maturity = valid.groupby("purchase_month").agg(
    orders=("order_id", "size"),
    not_delivered=("is_delivered", lambda s: 1 - s.mean()),
    no_review=("review_score", lambda s: s.isna().mean()),
)
print(maturity.tail(8).round(3))

                orders  not_delivered  no_review
purchase_month                                  
2018-01-01        7187          0.016      0.008
2018-02-01        6625          0.011      0.008
2018-03-01        7168          0.023      0.009
2018-04-01        6919          0.017      0.008
2018-05-01        6833          0.012      0.005
2018-06-01        6145          0.008      0.004
2018-07-01        6233          0.012      0.007
2018-08-01        6421          0.011      0.004


Checking maturity of the data with delivering rates.


In [10]:
import pandas as pd



seller_months = con.execute(f"""
    SELECT DISTINCT
        oi.seller_id,
        CAST(DATE_TRUNC('month', o.order_purchase_timestamp) AS DATE) AS month
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    WHERE o.order_status NOT IN ('canceled', 'unavailable')
      AND CAST(o.order_purchase_timestamp AS DATE) BETWEEN DATE '{START}' AND DATE '{END}'
""").df()
seller_months["month"] = pd.to_datetime(seller_months["month"])

MARTS.mkdir(parents=True, exist_ok=True)
facts.to_parquet(MARTS / "order_facts.parquet", index=False)
seller_months.to_parquet(MARTS / "seller_months.parquet", index=False)

for f in MARTS.glob("*.parquet"):
    print(f.name, f"{f.stat().st_size / 1e6:.1f} MB")

order_facts.parquet 7.6 MB
seller_months.parquet 0.1 MB


In [11]:
import pandas as pd
from src.metrics import monthly_kpis, first_two_purchases, repeat_rate_within, cohort_matrix

# Make sure imports succeed without error:
print("Imports OK!")


Imports OK!
